# Assignment 4 Data Analysis

- Name: Kenji Otsuka
- Email: kotsuka@umich.edu

In this notebook, I combine and analyze the `.fit` files from the `extra` folder.

## Adherence to Rule et al.'s "Ten Simple Rules for Writing and Sharing Computational Analyses in Jupyter Notebooks"

#### Rule 3: Use Cell Divisions to Make Steps Clear

I organized this notebooks so that each cell performs one meaningful step of the analysis or computation.
Markdown headers divide the notebook into clear sections.

## Load Ligraries

In [4]:
!python -m pip install --target=$HOME/.local fitparse

  Using cached fitparse-1.2.0-py3-none-any.whl


In [5]:
import sys
sys.path.append("/voc/work/.local")

In [6]:
import fitparse
from fitparse import FitFile
import os
import pandas as pd
from pathlib import Path

## Combine and Create One CSV file

Combine all `.fit` files and export as one file.

In [7]:
FIT_FILE_DIR = "/voc/coursework/resources/extra"

def fit_file_path(fname):
    return os.path.join(FIT_FILE_DIR, fname)

In [ ]:
out_dir = "csv_output"
os.makedirs(out_dir, exist_ok=True)
combined_dfs = []

for fname in os.listdir(FIT_FILE_DIR):
    if not fname.lower().endswith(".fit"):
        continue
    path = fit_file_path(fname)
    # print("Parsing:", path)
    fitfile = fitparse.FitFile(path)
    rows = []
    # each 'record' message typically contains time, position, altitude, speed, etc.
    for record in fitfile.get_messages("record"):
        data = {}
        for field in record:
            # field has .name and .value; skip None values
            try:
                name = field.name
                val = field.value
            except Exception:
                continue
            if name and val is not None:
                data[name] = val
        if data:
            rows.append(data)
    df = pd.DataFrame(rows)
    df["source_file"] = fname
    #per_csv = os.path.join(out_dir, fname + ".csv")
    #df.to_csv(per_csv, index=False)
    combined_dfs.append(df)

Combine all parsed records and save a single CSV

In [ ]:
df_all = pd.concat(combined_dfs, ignore_index=True, sort=False)
df_all.to_csv(os.path.join(out_dir, "combined.csv"), index=False)
print("Wrote combined CSV with rows:", len(df_all))

Wrote combined CSV with rows: 529985


## Analyze Unit of Measurement

Pick up a `.fit` file and analyze the unit of measurement for each field.

In [8]:
fit_path = fit_file_path('4491335236.fit')
fitfile = FitFile(str(fit_path))

See file information.

In [9]:
record_messages = list(fitfile.get_messages('record'))
print(f'file: {fit_path}')
print(f'record messages: {len(record_messages)}')

file: /voc/coursework/resources/extra/4491335236.fit
record messages: 7689


Get the first record and create a dataframe of feature information. This is for trial to confirm computation way.

In [10]:
first = record_messages[0]
rows = []
for field in first:
    rows.append({
        'name': getattr(field, 'name', None),
        'value': getattr(field, 'value', None),
        'units': getattr(field, 'units', None),
        'native_units': getattr(field, 'native_units', None),
        'scale': getattr(field, 'scale', None),
        'offset': getattr(field, 'offset', None),
        'defn_units': getattr(getattr(field, 'defn', None), 'units', None),
        'defn_scale': getattr(getattr(field, 'defn', None), 'scale', None),
        'defn_offset': getattr(getattr(field, 'defn', None), 'offset', None),
    })

df = pd.DataFrame(rows)
# display(df)

In [15]:
unit_map = (
    df[['name', 'units', 'native_units', 'defn_units']]
    .sort_values(['name', 'units', 'native_units', 'defn_units'], na_position='last')
)
display(unit_map)

,name,units,native_units,defn_units
0,Air Power,Watts,None,None
1,Cadence,RPM,None,None
2,Form Power,Watts,None,None
3,Ground Time,Milliseconds,None,None
4,Leg Spring Stiffness,kN/m,None,None
5,Power,Watts,None,None
6,Vertical Oscillation,Centimeters,None,None
7,cadence,rpm,None,None
8,distance,m,None,None
9,enhanced_speed,m/s,None,None


Build Unit data from all records in the file.

In [12]:
all_rows = []
for message in record_messages:
    for field in message:
        all_rows.append({
            'name': getattr(field, 'name', None),
            'units': getattr(field, 'units', None),
            'value': getattr(field, 'value', None),
        })

In [13]:
all_df = pd.DataFrame(all_rows)
summary = (
    all_df.groupby(['name', 'units'], dropna=False)
    .size()
    .reset_index(name='occurrences')
    .sort_values(['name', 'units'], na_position='last')
)
display(summary)

,name,units,occurrences
0,Air Power,Watts,7689
1,Cadence,RPM,7689
2,Form Power,Watts,7689
3,Ground Time,Milliseconds,7689
4,Leg Spring Stiffness,kN/m,7689
5,Power,Watts,7689
6,Vertical Oscillation,Centimeters,7689
7,cadence,rpm,7689
8,distance,m,7689
9,enhanced_altitude,m,7653


Calculating from all records, 19 kinds of values are found, while the first record has only 16.